# Supplementary Results 11.1-11.3 — Deriving and validating the combined criterion

Where the "PAV support in 2-5 therapeutic areas" criterion comes from, how sensitive its odds ratio
is to the window, and what it gives on targets that took no part in estimating it.

Numbers are written to `results/sr11_criterion.json`. Supplementary Results 11.4 and 11.5 — the
Pharmaprojects check and the interaction — are in `12_external_replication.ipynb`.

**Provenance.** `chapters/_legacy/06-review-r1/or10-optimism-validation/`, notebooks 02 and 03,
written in response to referee 2's major comment 2. The enrichment statistics and the support
definition are `manuscript_methods.enrichment`, which is that analysis's `or10_stats.py`.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2

from manuscript_methods import enrichment, paper

numbers = {}

N_SPLITS = 200
SEED = 20260811
PUBLISHED_WINDOW = (2, 5)

pairs = pd.read_parquet(paper.derived("ti_pairs_chembl"))
print(f"target-indication pairs: {len(pairs):,} | approved: {int(pairs['approved'].sum()):,}")

target-indication pairs: 37,377 | approved: 4,564


## The criterion itself

The two results it follows from: PAV support carries higher enrichment than non-PAV support, and
enrichment declines with pleiotropy along a curve peaking below two therapeutic areas.

In [2]:
baseline = enrichment.or_rs(enrichment.support_mask(pairs), pairs["approved"])
criterion = enrichment.or_rs(enrichment.support_mask(pairs, pav=True, ta_min=2, ta_max=5), pairs["approved"])
BASELINE_OR = baseline["odds_ratio"]

numbers["S11.01"] = criterion["n_support"]
numbers["S11.02"] = criterion["yes_evid-high_clinphase"]
numbers["S11.03"] = round(100 * criterion["yes_evid-high_clinphase"] / baseline["yes_evid-high_clinphase"], 1)
print(f"all-GWAS support: OR {BASELINE_OR:.4f}, {baseline['yes_evid-high_clinphase']} approved pairs")
print(f"criterion: {numbers['S11.01']} pairs, {numbers['S11.02']} approved, {numbers['S11.03']}% of the 242")

all-GWAS support: OR 3.6186, 242 approved pairs
criterion: 87 pairs, 51 approved, 21.1% of the 242


In [3]:
# Each stratum is compared against the unsupported pairs only, so the two supported strata are not
# each other's control. That is what gives the published 6.05 and 3.09; taking every other pair as
# the control instead gives 5.89 and 3.05.
pav = enrichment.support_mask(pairs, pav=True)
non_pav = enrichment.support_mask(pairs) & ~pav
unsupported = ~enrichment.support_mask(pairs)


def stratum(mask):
    """Enrichment of one supported stratum against the unsupported pairs."""
    subset = pairs[mask | unsupported]
    return enrichment.or_rs(mask.loc[subset.index], subset["approved"])


pav_result = stratum(pav)
non_pav_result = stratum(non_pav)

difference = enrichment.contrast(
    pav_result["yes_evid-high_clinphase"],
    pav_result["yes_evid-low_clinphase"],
    non_pav_result["yes_evid-high_clinphase"],
    non_pav_result["yes_evid-low_clinphase"],
)
numbers["S11.04"] = round(pav_result["odds_ratio"], 2)
numbers["S11.05"] = round(non_pav_result["odds_ratio"], 2)
numbers["S11.06"] = difference["p_value"]
print(
    f"PAV support OR {numbers['S11.04']} against non-PAV OR {numbers['S11.05']}, difference P {numbers['S11.06']:.2e}"
)

PAV support OR 6.05 against non-PAV OR 3.09, difference P 2.44e-04


## Threshold sensitivity

Every therapeutic-area window crossed with PAV status. A window is reportable when it carries at
least ten approved supported pairs.

In [4]:
def evaluate(frame, pav, ta_min, ta_max):
    """Odds ratio, relative success and counts for one support definition."""
    mask = enrichment.support_mask(frame, pav=pav, ta_min=ta_min, ta_max=ta_max)
    return {
        "pav": pav,
        "ta_min": ta_min,
        "ta_max": ta_max,
        "window": enrichment.window_label(ta_min, ta_max),
        **enrichment.or_rs(mask, frame["approved"]),
    }


windows = [(lo, hi) for lo in range(1, 9) for hi in list(range(1, 13)) + [None] if hi is None or hi >= lo]
grid = pd.DataFrame([evaluate(pairs, pav, lo, hi) for pav in (True, False) for lo, hi in windows])
usable = (
    grid[(grid["pav"]) & (grid["yes_evid-high_clinphase"] >= 10)]
    .sort_values("odds_ratio", ascending=False)
    .reset_index(drop=True)
)
usable["rank"] = usable.index + 1
published_row = usable[(usable["ta_min"] == 2) & (usable["ta_max"] == 5)].iloc[0]

numbers["S11.07"] = len(usable)
numbers["S11.08"] = int(published_row["rank"])
numbers["S11.09"] = int(
    ((usable["odds_ratio"] >= published_row["ci_low"]) & (usable["odds_ratio"] <= published_row["ci_high"])).sum()
)
numbers["S11.10"] = int(usable["ci_low"].gt(BASELINE_OR).sum())
numbers["S11.11"] = int((usable["odds_ratio"] > published_row["odds_ratio"]).sum())
print(f"reportable PAV windows: {numbers['S11.07']} | rank of 2-5: {numbers['S11.08']}")
print(f"inside the published CI [{published_row['ci_low']:.2f}, {published_row['ci_high']:.2f}]: {numbers['S11.09']}")
print(f"CI excluding the baseline: {numbers['S11.10']} | point estimate above 2-5: {numbers['S11.11']}")
usable.head(6)[["rank", "window", "odds_ratio", "ci_low", "ci_high", "yes_evid-high_clinphase"]].round(3)

reportable PAV windows: 64 | rank of 2-5: 4
inside the published CI [6.71, 15.78]: 32
CI excluding the baseline: 45 | point estimate above 2-5: 3


,rank,window,odds_ratio,ci_low,ci_high,yes_evid-high_clinphase
0,1,2,18.011,5.646,57.452,10
1,2,2-3,10.823,5.210,22.484,18
2,3,2-4,10.353,6.381,16.797,40
3,4,2-5,10.289,6.708,15.782,51
4,5,4-5,9.950,5.876,16.850,33
5,6,4,9.929,5.211,18.919,22


In [5]:
named = pd.DataFrame(
    [evaluate(pairs, True, lo, hi) for lo, hi in [(2, 2), (2, 3), (2, 4), (1, 5), (3, 5), (2, 6), (2, 5)]]
)
lookup = named.set_index("window")
numbers["S11.12"] = round(float(lookup.loc["2", "odds_ratio"]), 2)
numbers["S11.13"] = int(lookup.loc["2", "yes_evid-high_clinphase"])
numbers["S11.14"] = round(float(lookup.loc["2-3", "odds_ratio"]), 2)
numbers["S11.15"] = int(lookup.loc["2-3", "yes_evid-high_clinphase"])
numbers["S11.16"] = round(float(lookup.loc["2-4", "odds_ratio"]), 2)
numbers["S11.17"] = int(lookup.loc["2-4", "yes_evid-high_clinphase"])
numbers["S11.18"] = round(float(lookup.loc["1-5", "odds_ratio"]), 2)
numbers["S11.19"] = round(float(lookup.loc["3-5", "odds_ratio"]), 2)
numbers["S11.20"] = round(float(lookup.loc["2-6", "odds_ratio"]), 2)
named[["window", "odds_ratio", "ci_low", "ci_high", "relative_success", "n_support", "yes_evid-high_clinphase"]].round(
    3
)

,window,odds_ratio,ci_low,ci_high,relative_success,n_support,yes_evid-high_clinphase
0,2,18.011,5.646,57.452,5.860,14,10
1,2-3,10.823,5.210,22.484,4.929,30,18
2,2-4,10.353,6.381,16.797,4.851,68,40
3,1-5,8.992,5.981,13.520,4.571,94,52
4,3-5,9.286,5.842,14.759,4.632,73,41
5,2-6,9.467,6.355,14.104,4.678,99,56
6,2-5,10.289,6.708,15.782,4.844,87,51


## What each component contributes

Supplementary Materials Table `sr_or10_decomposition`.

In [6]:
decomposition = pd.DataFrame(
    [
        {"definition": "All GWAS support", **enrichment.or_rs(enrichment.support_mask(pairs), pairs["approved"])},
        {
            "definition": "PAV, any TA count",
            **enrichment.or_rs(enrichment.support_mask(pairs, pav=True), pairs["approved"]),
        },
        {
            "definition": "Any support, 2-5 TAs",
            **enrichment.or_rs(enrichment.support_mask(pairs, ta_min=2, ta_max=5), pairs["approved"]),
        },
        {
            "definition": "Any support, >=2 TAs",
            **enrichment.or_rs(enrichment.support_mask(pairs, ta_min=2), pairs["approved"]),
        },
        {
            "definition": "PAV, 2-5 TAs",
            **enrichment.or_rs(enrichment.support_mask(pairs, pav=True, ta_min=2, ta_max=5), pairs["approved"]),
        },
    ]
)
for offset, (_, row) in enumerate(decomposition.iterrows()):
    numbers[f"S11.{21 + 3 * offset:02d}"] = round(float(row["odds_ratio"]), 2)
    numbers[f"S11.{22 + 3 * offset:02d}"] = round(float(row["relative_success"]), 2)
    numbers[f"S11.{23 + 3 * offset:02d}"] = int(row["yes_evid-high_clinphase"])

pav_multiplier = decomposition.loc[1, "odds_ratio"] / BASELINE_OR
ta_multiplier = decomposition.loc[2, "odds_ratio"] / BASELINE_OR
numbers["S11.36"] = round(float(pav_multiplier), 2)
numbers["S11.37"] = round(float(ta_multiplier), 2)
numbers["S11.38"] = round(float(BASELINE_OR * pav_multiplier * ta_multiplier), 1)
print(f"multipliers: PAV {numbers['S11.36']}, therapeutic-area window {numbers['S11.37']}")
print(f"if they acted independently: OR {numbers['S11.38']}")
decomposition[["definition", "odds_ratio", "relative_success", "yes_evid-high_clinphase"]].round(2)

multipliers: PAV 1.63, therapeutic-area window 1.09
if they acted independently: OR 6.4


,definition,odds_ratio,relative_success,yes_evid-high_clinphase
0,All GWAS support,3.62,2.76,242
1,"PAV, any TA count",5.89,3.71,72
2,"Any support, 2-5 TAs",3.95,2.92,139
3,"Any support, >=2 TAs",3.54,2.72,220
4,"PAV, 2-5 TAs",10.29,4.84,51


## Held-out validation

200 splits, always by target and never by pair, stratified on the number of approved pairs each
target carries. The criterion is held fixed at PAV and 2-5 therapeutic areas and evaluated on the
half that took no part in choosing it.

In [7]:
pairs["ta"] = pairs["uniqueTherapeuticAreas"].fillna(0).astype(float)
pairs["support_all"] = enrichment.support_mask(pairs).astype(int)
pairs["support_pav"] = enrichment.support_mask(pairs, pav=True).astype(int)
approved_per_target = pairs.groupby("targetId")["approved"].sum()


def target_split(generator):
    """Assign every target to one half or the other, stratified on approved-pair count."""
    half_a, half_b = [], []
    for _, targets in approved_per_target.groupby(approved_per_target.values):
        ids = targets.index.to_numpy()
        generator.shuffle(ids)
        start = generator.integers(2)
        for position, target in enumerate(ids):
            (half_a if (position + start) % 2 == 0 else half_b).append(target)
    return set(half_a), set(half_b)


generator = np.random.default_rng(SEED)
splits = []
for _ in range(N_SPLITS):
    a_ids, _ = target_split(generator)
    in_a = pairs["targetId"].isin(a_ids).to_numpy()
    splits.append((pairs[in_a], pairs[~in_a]))
print(
    f"splits: {len(splits)} | median approved pairs per half: "
    f"{np.median([int(b['approved'].sum()) for _, b in splits]):.0f}"
)

splits: 200 | median approved pairs per half: 2286


In [8]:
rows = []
for index, (half_a, half_b) in enumerate(splits):
    row = {"split": index}
    for name, half in (("a", half_a), ("b", half_b)):
        result = enrichment.or_rs(
            enrichment.support_mask(half, pav=True, ta_min=PUBLISHED_WINDOW[0], ta_max=PUBLISHED_WINDOW[1]),
            half["approved"],
        )
        degenerate = result["yes_evid-high_clinphase"] == 0 or result["yes_evid-low_clinphase"] == 0
        row[f"or_{name}"] = np.nan if degenerate else result["odds_ratio"]
        row[f"ci_low_{name}"] = result["ci_low"]
        row[f"approved_{name}"] = result["yes_evid-high_clinphase"]
    rows.append(row)
held_out = pd.DataFrame(rows)

usable_pairs = held_out[["or_a", "or_b"]].replace([np.inf, -np.inf], np.nan).dropna()
log_difference = np.log(usable_pairs["or_a"]) - np.log(usable_pairs["or_b"])
mcse = float(log_difference.std(ddof=1) / np.sqrt(len(log_difference)))

numbers["S11.39"] = round(float(held_out["or_b"].median()), 2)
numbers["S11.40"] = round(float(np.nanpercentile(held_out["or_b"], 2.5)), 2)
numbers["S11.41"] = round(float(np.nanpercentile(held_out["or_b"], 97.5)), 2)
numbers["S11.42"] = round(float(np.exp(log_difference.mean())), 2)
numbers["S11.43"] = round(float(np.exp(log_difference.mean() - 1.96 * mcse)), 2)
numbers["S11.44"] = round(float(np.exp(log_difference.mean() + 1.96 * mcse)), 2)
numbers["S11.45"] = round(100 * float((held_out["ci_low_b"] > BASELINE_OR).mean()), 1)
print(f"held-out median OR {numbers['S11.39']}, spread {numbers['S11.40']}-{numbers['S11.41']}")
print(f"in-sample over held-out ratio {numbers['S11.42']} ({numbers['S11.43']}-{numbers['S11.44']})")
print(f"splits whose held-out CI excludes the baseline: {numbers['S11.45']}%")

held-out median OR 10.32, spread 6.62-16.75
in-sample over held-out ratio 0.97 (0.91-1.04)
splits whose held-out CI excludes the baseline: 95.5%


## Do the two founding results survive in half-sized samples

In [9]:
def half_checks(half):
    """PAV against non-PAV, and the quadratic likelihood-ratio test, on one half."""
    pav_mask = enrichment.support_mask(half, pav=True)
    non_pav_mask = enrichment.support_mask(half) & ~pav_mask
    pav_side = enrichment.or_rs(pav_mask, half["approved"])
    non_pav_side = enrichment.or_rs(non_pav_mask, half["approved"])
    difference = enrichment.contrast(
        pav_side["yes_evid-high_clinphase"],
        pav_side["yes_evid-low_clinphase"],
        non_pav_side["yes_evid-high_clinphase"],
        non_pav_side["yes_evid-low_clinphase"],
    )
    quadratic = smf.logit("approved ~ support_all + I(np.log(ta+1)) + I(np.log(ta+1) ** 2)", data=half).fit(disp=False)
    linear = smf.logit("approved ~ support_all + I(np.log(ta+1))", data=half).fit(disp=False)
    statistic = 2 * (float(quadratic.llf) - float(linear.llf))
    return {
        "pav_higher": pav_side["odds_ratio"] > non_pav_side["odds_ratio"],
        "pav_p": difference["p_value"],
        "quadratic_p": float(chi2.sf(statistic, 1)),
    }


checks = pd.DataFrame([half_checks(half_b) for _, half_b in splits])
numbers["S11.46"] = round(100 * float(checks["pav_higher"].mean()), 1)
numbers["S11.47"] = round(100 * float((checks["pav_p"] < 0.05).mean()), 1)
numbers["S11.48"] = round(100 * float((checks["quadratic_p"] < 0.05).mean()), 1)
numbers["S11.49"] = round(100 * float((checks["quadratic_p"] < 1e-4).mean()), 1)
print({k: numbers[k] for k in ["S11.46", "S11.47", "S11.48", "S11.49"]})

{'S11.46': 100.0, 'S11.47': 79.0, 'S11.48': 99.5, 'S11.49': 89.0}


## Optimism from searching for the best window

The window that maximises enrichment on one half, applied unchanged to the other. The search is
restricted to windows wide enough and populated enough to report.

In [10]:
CANDIDATES = [(lo, hi) for lo in range(1, 7) for hi in list(range(lo, 11)) + [None]]
MIN_APPROVED, MIN_WIDTH = 20, 3


def width(lo, hi):
    """Number of therapeutic-area values a window admits; unbounded windows count as wide."""
    return 99 if hi is None else hi - lo + 1


search_rows = []
for index, (half_a, half_b) in enumerate(splits):
    best = None
    for lo, hi in CANDIDATES:
        if width(lo, hi) < MIN_WIDTH:
            continue
        result = enrichment.or_rs(enrichment.support_mask(half_a, pav=True, ta_min=lo, ta_max=hi), half_a["approved"])
        if result["yes_evid-high_clinphase"] < MIN_APPROVED or result["yes_evid-low_clinphase"] == 0:
            continue
        if best is None or result["odds_ratio"] > best[0]:
            best = (result["odds_ratio"], lo, hi)
    if best is None:
        continue
    or_a, lo, hi = best
    held = enrichment.or_rs(enrichment.support_mask(half_b, pav=True, ta_min=lo, ta_max=hi), half_b["approved"])
    if held["yes_evid-high_clinphase"] == 0 or held["yes_evid-low_clinphase"] == 0:
        continue
    rs_a = enrichment.or_rs(enrichment.support_mask(half_a, pav=True, ta_min=lo, ta_max=hi), half_a["approved"])[
        "relative_success"
    ]
    search_rows.append(
        {
            "split": index,
            "window": enrichment.window_label(lo, hi),
            "or_a": or_a,
            "or_b": held["odds_ratio"],
            "rs_a": rs_a,
            "rs_b": held["relative_success"],
        }
    )

search = pd.DataFrame(search_rows)
optimism = float(np.exp((np.log(search["or_a"]) - np.log(search["or_b"])).mean()))
numbers["S11.50"] = round(optimism, 2)
numbers["S11.51"] = round(float(criterion["odds_ratio"] / optimism), 2)
numbers["S11.52"] = round(float(criterion["ci_low"] / optimism), 2)
numbers["S11.53"] = round(float(criterion["ci_high"] / optimism), 2)
rs_optimism = float(np.exp((np.log(search["rs_a"]) - np.log(search["rs_b"])).mean()))
numbers["S11.54"] = round(float(criterion["relative_success"] / rs_optimism), 2)
print(f"splits contributing: {len(search)} | optimism factor {numbers['S11.50']}")
print(f"corrected OR {numbers['S11.51']} ({numbers['S11.52']}-{numbers['S11.53']})")
print(f"most frequently selected windows:")
print(search["window"].value_counts().head(5).to_string())

splits contributing: 198 | optimism factor 1.19
corrected OR 8.63 (5.63-13.24)
most frequently selected windows:
window
2-5    69
2-4    61
4-6    11
2-7    11
4-7    10


## Write the results

In [11]:
print(paper.save_results("sr11_criterion", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr11_criterion.json


,computed
S11.01,87.000000
S11.02,51.000000
S11.03,21.100000
S11.04,6.050000
S11.05,3.090000
S11.06,0.000244
S11.07,64.000000
S11.08,4.000000
S11.09,32.000000
S11.10,45.000000
